Write a Python script to fetch user data from a public REST API, clean it, and merge it with a flat CSV file containing location details.

In [1]:
import pandas as pd
import requests

def extract_api_data(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        return pd.json_normalize(data)
    except requests.exceptions.RequestException as e:
        print(f"API Error: {e}")
        return pd.DataFrame()

def extract_csv_data(file_path):
    try:
        return pd.read_csv(file_path)
    except FileNotFoundError as e:
        print(f"File Error: {e}")
        return pd.DataFrame()

In [2]:
api_url = "https://jsonplaceholder.typicode.com/users"
api_df = extract_api_data(api_url)
if not api_df.empty:
    api_df = api_df[["id", "name", "email", "company.name"]]
    api_df.rename(columns={"company.name": "company"}, inplace=True)

csv_file = "locations.csv"
pd.DataFrame(
    {
        "id": list(range(1, 11)),
        "city": [
            "New York",
            "London",
            "Paris",
            "Tokyo",
            "Berlin",
            "Delhi",
            "Sydney",
            "Moscow",
            "Cairo",
            "Beijing",
        ],
    }
).to_csv(csv_file, index=False)

csv_df = extract_csv_data(csv_file)

if not api_df.empty and not csv_df.empty:
    merged_df = pd.merge(api_df, csv_df, on="id", how="inner")
    print("\n--- Merged ETL Pipeline Data View ---")
    display(merged_df.head())
    merged_df.to_csv("cleaned_warehouse_profiles.csv", index=False)
    print("\nData successfully saved to 'cleaned_warehouse_profiles.csv'")


--- Merged ETL Pipeline Data View ---


,id,name,email,company,city
0,1,Leanne Graham,Sincere@april.biz,Romaguera-Crona,New York
1,2,Ervin Howell,Shanna@melissa.tv,Deckow-Crist,London
2,3,Clementine Bauch,Nathan@yesenia.net,Romaguera-Jacobson,Paris
3,4,Patricia Lebsack,Julianne.OConner@kory.org,Robel-Corkery,Tokyo
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Keebler LLC,Berlin



Data successfully saved to 'cleaned_warehouse_profiles.csv'
